# HTML Page Modifier with LangGraph

LangGraph 기반 단일 HTML 페이지 수정 워크플로우

## Features
- 단일 HTML 페이지 입력 → AI 기반 수정 → 수정된 HTML 출력
- 사용자 요청에 따른 디자인/레이아웃 수정
- Gemini / Azure OpenAI 지원

## Workflow
```
START → modify_page_html → END
```

## Cell 1: Setup & Imports

In [1]:
# Standard library
from typing import Literal
from typing_extensions import TypedDict
import os

# LangGraph
from langgraph.graph import StateGraph, START, END

# LangChain LLMs
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import AzureChatOpenAI

# Utilities
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

print("Imports loaded successfully!")

Imports loaded successfully!


## Cell 2: Input & State Schema Definition

In [2]:
# =============================================================================
# Input Schema - 페이지 수정 입력 구조
# =============================================================================

class PageModificationInput(TypedDict):
    """페이지 수정 입력 스키마"""
    html_content: str      # 현재 HTML 콘텐츠
    user_request: str      # 사용자 수정 요청
    page_id: str           # 페이지 식별자
    page_order: int        # 페이지 순서


# =============================================================================
# Workflow State Schema
# =============================================================================

class PageModificationState(TypedDict):
    """워크플로우 상태 스키마"""
    
    # 입력
    input: PageModificationInput
    
    # 출력
    modified_html: str           # 수정된 HTML
    modification_summary: str    # 수정 내용 요약
    
    # 상태
    status: str


print("State schemas defined!")
print(f"- PageModificationInput: html_content, user_request, page_id, page_order")
print(f"- PageModificationState: input, modified_html, modification_summary, status")

State schemas defined!
- PageModificationInput: html_content, user_request, page_id, page_order
- PageModificationState: input, modified_html, modification_summary, status


## Cell 3: LLM Configuration

In [3]:
# =============================================================================
# LLM Provider Configuration
# =============================================================================

def get_gemini_llm(temperature: float = 0.3) -> ChatGoogleGenerativeAI:
    """Google Gemini LLM 초기화"""
    return ChatGoogleGenerativeAI(
        model=os.getenv("GEMINI_MODEL", "gemini-2.0-flash"),
        google_api_key=os.getenv("GOOGLE_API_KEY"),
        temperature=temperature
    )


def get_azure_llm(temperature: float = 0.3) -> AzureChatOpenAI:
    """Azure OpenAI LLM 초기화"""
    return AzureChatOpenAI(
        azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),
        api_version=os.getenv("OPENAI_API_VERSION", "2024-08-01-preview"),
        deployment_name="gpt-4o",
        temperature=temperature
    )


def get_llm(provider: Literal["gemini", "azure"] = "gemini", temperature: float = 0.3):
    """
    LLM Factory Function
    
    Args:
        provider: "gemini" 또는 "azure"
        temperature: 생성 온도 (0.0-1.0), HTML 수정은 낮은 값 권장
    
    Returns:
        LLM instance
    """
    if provider == "gemini":
        return get_gemini_llm(temperature)
    elif provider == "azure":
        return get_azure_llm(temperature)
    else:
        raise ValueError(f"Unknown provider: {provider}. Use 'gemini' or 'azure'.")


# Default LLM provider setting
LLM_PROVIDER: Literal["gemini", "azure"] = "gemini"

print(f"LLM Configuration ready!")
print(f"- Default provider: {LLM_PROVIDER}")
print(f"- Default temperature: 0.3 (deterministic for HTML)")

LLM Configuration ready!
- Default provider: gemini
- Default temperature: 0.3 (deterministic for HTML)


## Cell 4: Helper Functions

In [4]:
# =============================================================================
# Helper Functions
# =============================================================================

def extract_text_content(response_content) -> str:
    """
    LLM 응답에서 텍스트 콘텐츠를 추출합니다.
    일부 LLM(Gemini 등)은 content를 리스트로 반환할 수 있습니다.
    """
    if isinstance(response_content, str):
        return response_content
    elif isinstance(response_content, list):
        text_parts = []
        for item in response_content:
            if isinstance(item, str):
                text_parts.append(item)
            elif isinstance(item, dict) and "text" in item:
                text_parts.append(item["text"])
            elif hasattr(item, "text"):
                text_parts.append(item.text)
        return "".join(text_parts)
    else:
        return str(response_content)


def extract_html_content(response_content) -> str:
    """
    LLM 응답에서 HTML 콘텐츠를 추출합니다.
    마크다운 코드 블록(```html ... ```)을 제거합니다.
    """
    content = extract_text_content(response_content)
    
    # Remove markdown code blocks
    if "```html" in content:
        content = content.split("```html")[1].split("```")[0]
    elif "```" in content:
        parts = content.split("```")
        if len(parts) >= 2:
            content = parts[1]
            # Remove language identifier if present
            if content.startswith("html\n"):
                content = content[5:]
    
    return content.strip()


print("Helper functions defined!")

Helper functions defined!


## Cell 5: Page Modification Node

In [5]:
# =============================================================================
# Node: Page HTML Modification
# =============================================================================

def modify_page_html(state: PageModificationState) -> dict:
    """
    HTML 페이지 수정 노드
    
    사용자 요청에 따라 HTML 콘텐츠를 수정합니다.
    
    Returns:
        dict: {"modified_html": str, "modification_summary": str, "status": str}
    """
    llm = get_llm(LLM_PROVIDER, temperature=0.3)
    user_input = state["input"]
    
    prompt = f"""
당신은 HTML/CSS 디자인 전문가입니다.
주어진 HTML을 사용자 요청에 따라 수정하세요.

## 현재 페이지 정보
- Page ID: {user_input["page_id"]}
- Page Order: {user_input["page_order"]}

## 현재 HTML
```html
{user_input["html_content"]}
```

## 사용자 수정 요청
{user_input["user_request"]}

## 지침
1. 사용자 요청에 정확히 맞게 HTML을 수정하세요.
2. 기존 HTML 구조를 최대한 유지하면서 필요한 부분만 수정하세요.
3. CSS 스타일은 inline style 또는 <style> 태그 내에 포함하세요.
4. 수정된 HTML만 반환하세요. 설명이나 추가 텍스트는 포함하지 마세요.

수정된 HTML:
"""
    
    response = llm.invoke(prompt)
    modified_html = extract_html_content(response.content)
    
    # Generate modification summary
    summary_prompt = f"""
다음 HTML 수정 요청에 대해 수행된 변경 사항을 1-2문장으로 간략히 요약하세요.

수정 요청: {user_input["user_request"]}

요약 (한국어로):
"""
    
    summary_response = llm.invoke(summary_prompt)
    summary = extract_text_content(summary_response.content).strip()
    
    print(f"페이지 수정 완료: {user_input['page_id']} (order: {user_input['page_order']})")
    print(f"수정 요약: {summary}")
    
    return {
        "modified_html": modified_html,
        "modification_summary": summary,
        "status": "completed"
    }


print("Page Modification Node defined!")

Page Modification Node defined!


## Cell 6: Graph Construction

In [6]:
# =============================================================================
# LangGraph Workflow Construction
# =============================================================================

def create_page_modifier():
    """
    페이지 수정 워크플로우 그래프를 구성하고 컴파일합니다.
    
    Workflow:
        START -> modify_page_html -> END
    
    Returns:
        CompiledGraph: 실행 가능한 그래프
    """
    # StateGraph 초기화
    builder = StateGraph(PageModificationState)
    
    # 노드 추가
    builder.add_node("modify_page_html", modify_page_html)
    
    # 엣지 연결
    builder.add_edge(START, "modify_page_html")
    builder.add_edge("modify_page_html", END)
    
    # 그래프 컴파일
    graph = builder.compile()
    
    print("Page Modifier Graph compiled!")
    print("Workflow: START -> modify_page_html -> END")
    
    return graph


# 그래프 생성
page_modifier = create_page_modifier()

Page Modifier Graph compiled!
Workflow: START -> modify_page_html -> END


## Cell 7: Test Execution

In [7]:
# =============================================================================
# 샘플 테스트
# =============================================================================

# 테스트용 샘플 HTML
sample_html = """
<div class="page-container">
    <header>
        <h1>주간 업무 보고서</h1>
        <p class="date">2024년 4분기</p>
    </header>
    
    <section class="summary">
        <h2>요약</h2>
        <p>이번 주 마케팅 캠페인 성과가 우수했습니다.</p>
        <ul>
            <li>클릭률 15% 상승</li>
            <li>전환율 3.2% 달성</li>
        </ul>
    </section>
</div>
"""

# 테스트 입력 데이터
sample_input: PageModificationInput = {
    "html_content": sample_html,
    "user_request": "헤더 배경색을 파란색(#3498db)으로 변경하고, 제목을 흰색으로 만들어주세요. 요약 섹션에 연한 회색 배경을 추가해주세요.",
    "page_id": "weekly_summary",
    "page_order": 1
}

print("Sample input configured:")
print(f"  - page_id: {sample_input['page_id']}")
print(f"  - page_order: {sample_input['page_order']}")
print(f"  - user_request: {sample_input['user_request']}")

Sample input configured:
  - page_id: weekly_summary
  - page_order: 1
  - user_request: 헤더 배경색을 파란색(#3498db)으로 변경하고, 제목을 흰색으로 만들어주세요. 요약 섹션에 연한 회색 배경을 추가해주세요.


In [8]:
# =============================================================================
# 워크플로우 실행
# =============================================================================

# 초기 상태 구성
initial_state: PageModificationState = {
    "input": sample_input,
    "modified_html": "",
    "modification_summary": "",
    "status": "pending"
}

print("Starting page modification...")
print("="*50)

# 그래프 실행
result = page_modifier.invoke(initial_state)

print("="*50)
print(f"\nModification completed!")
print(f"Status: {result['status']}")
print(f"Summary: {result['modification_summary']}")

Starting page modification...
페이지 수정 완료: weekly_summary (order: 1)
수정 요약: 헤더의 배경색을 파란색(#3498db)으로, 제목을 흰색으로 변경하였습니다. 또한 요약 섹션에 연한 회색 배경을 추가하여 스타일을 수정했습니다.

Modification completed!
Status: completed
Summary: 헤더의 배경색을 파란색(#3498db)으로, 제목을 흰색으로 변경하였습니다. 또한 요약 섹션에 연한 회색 배경을 추가하여 스타일을 수정했습니다.


In [9]:
# =============================================================================
# 결과 저장
# =============================================================================

import os
from pathlib import Path

# Create test-files directory
output_dir = Path("test-files")
output_dir.mkdir(exist_ok=True)

# File paths
page_id = sample_input["page_id"]
original_file = output_dir / f"{page_id}_original.html"
modified_file = output_dir / f"{page_id}_modified.html"

# Save original HTML
with open(original_file, "w", encoding="utf-8") as f:
    f.write(sample_input["html_content"])

# Save modified HTML
with open(modified_file, "w", encoding="utf-8") as f:
    f.write(result["modified_html"])

print("Files saved successfully!")
print(f"  - Original: {original_file}")
print(f"  - Modified: {modified_file}")

Files saved successfully!
  - Original: test-files/weekly_summary_original.html
  - Modified: test-files/weekly_summary_modified.html


## Cell 8: Reusable Function

In [10]:
# =============================================================================
# 재사용 가능한 페이지 수정 함수
# =============================================================================

def modify_html(
    html_content: str,
    user_request: str,
    page_id: str,
    page_order: int,
    provider: Literal["gemini", "azure"] = "gemini"
) -> dict:
    """
    HTML 페이지를 수정합니다.
    
    Args:
        html_content: 현재 HTML 콘텐츠
        user_request: 사용자 수정 요청
        page_id: 페이지 식별자
        page_order: 페이지 순서
        provider: LLM 프로바이더 (선택)
    
    Returns:
        dict: {
            "modified_html": str,
            "modification_summary": str,
            "status": str
        }
    """
    global LLM_PROVIDER
    LLM_PROVIDER = provider
    
    user_input: PageModificationInput = {
        "html_content": html_content,
        "user_request": user_request,
        "page_id": page_id,
        "page_order": page_order
    }
    
    initial_state: PageModificationState = {
        "input": user_input,
        "modified_html": "",
        "modification_summary": "",
        "status": "pending"
    }
    
    result = page_modifier.invoke(initial_state)
    
    return {
        "modified_html": result["modified_html"],
        "modification_summary": result["modification_summary"],
        "status": result["status"]
    }


print("modify_html() function ready!")
print("\nUsage example:")
print('result = modify_html(')
print('    html_content="<div>...</div>",')
print('    user_request="헤더 색상을 파란색으로 변경해주세요",')
print('    page_id="summary",')
print('    page_order=1')
print(')')

modify_html() function ready!

Usage example:
result = modify_html(
    html_content="<div>...</div>",
    user_request="헤더 색상을 파란색으로 변경해주세요",
    page_id="summary",
    page_order=1
)
